In [14]:
from datasets import load_dataset
import json
import pandas as pd

# Load squall
file_path = "/home/raphael.gervillie/sql_graph/data/squall.json"
with open(file_path, 'r') as json_file:
    squall = json.load(json_file)

# Extract unique squall IDs
squall_ids = set(i["nt"] for i in squall)

# Load WTQ
wtq = load_dataset('wikitablequestions')
wtq_train = wtq["train"]


Using the latest cached version of the module from /home/raphael.gervillie/.cache/huggingface/modules/datasets_modules/datasets/wikitablequestions/4845d83f334ed5e4a8e0420731e64d57f696feb62e7d3a47b84037864fb8317c (last modified on Fri Feb 14 07:32:12 2025) since it couldn't be found locally at wikitablequestions, or remotely on the Hugging Face Hub.


In [15]:
# Extract WTQ IDs
wtq_ids = set(wtq_train["id"])

# Get intersection of IDs
common_ids = list(squall_ids.intersection(wtq_ids))

# Mapping from ID to table for squall
squall_table_id_by_id = {entry["nt"]: entry["tbl"] for entry in squall if entry["nt"] in common_ids}

# Mapping from ID to table for wtq
wtq_table_by_id = {entry["id"]: entry["table"] for entry in wtq_train if entry["id"] in common_ids}

In [16]:
id0 = common_ids[0]
durty_table = wtq_table_by_id[id0]
durty_table = pd.DataFrame(durty_table['rows'], columns=durty_table['header'])
durty_table

,Year,Game,Developer,Setting,Platform,Notes
0,1963,Intopia,,Modern,Various,
1,1973,Lemonade Stand,MECC,Modern,"Mainframe, APPII","Created in 1973, ported to Apple II in 1979"
2,1980,Windfall: The Oil Crisis Game,David Mullich,Modern,APPII,
3,1982,Airline,CCS,Modern,"ZX, BBC",
4,1982,Autochef,CCS,Modern,ZX,
...,...,...,...,...,...,...
275,TBA,Merchant,,Fantasy,"WIN, MAC, iOS",
276,TBA,The Business Sim,Various,Modern,WIN,
277,2012,Prison Architect,Introversion,Modern,WIN MAC LIN,In Alpha
278,2011,Cars Incorporated,David Klande,Historical,WIN,Alpha-version available


In [17]:
import sqlite3
import pandas as pd
import os

# Get the database ID from wtq
table_db_id = squall_table_id_by_id[id0]  # This gives you something like 'some_table_id'

# Path to the SQLite .db file
db_path = os.path.join("/home/raphael.gervillie/deep_sql/data/tables/db", f"{table_db_id}.db")
print(db_path)
# Connect to the SQLite database
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM w", conn)
conn.close()


/home/raphael.gervillie/deep_sql/data/tables/db/204_121.db


In [18]:
df

,id,agg,c1,c1_number,c1_parsed,c1_year,c2,c3,c3_length,c4,c5,c5_length,c6
0,1,0,1963,1963.0,1963,1963.0,intopia,None,0,modern,various,1,None
1,2,0,1973,1973.0,1973,1973.0,lemonade stand,mecc,1,modern,"mainframe, appii",2,"created in 1973, ported to apple ii in 1979"
2,3,0,1980,1980.0,1980,1980.0,windfall: the oil crisis game,david mullich,1,modern,appii,1,None
3,4,0,1982,1982.0,1982,1982.0,airline,ccs,1,modern,"zx, bbc",2,None
4,5,0,1982,1982.0,1982,1982.0,autochef,ccs,1,modern,zx,1,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,275,0,None,NaN,None,NaN,tycoon online,None,0,modern,win,1,None
275,276,0,None,NaN,None,NaN,merchant,None,0,fantasy,"win, mac, ios",3,None
276,277,0,None,NaN,None,NaN,the business sim,various,1,modern,win,1,None
277,278,0,2012,2012.0,2012,2012.0,prison architect,introversion,1,modern,win mac lin,1,in alpha


In [19]:
total_ids = range(df.shape[0])


In [21]:
import random

In [24]:
random_ids = random.sample(total_ids, 11)

In [25]:
df.iloc[random_ids]

,id,agg,c1,c1_number,c1_parsed,c1_year,c2,c3,c3_length,c4,c5,c5_length,c6
11,12,0,1982,1982.0,1982,1982.0,corn cropper,ccs,1,modern,zx,1,None
145,146,0,2003,2003.0,2003,2003.0,car tycoon,vectorcom,1,modern,win,1,None
21,22,0,1987,1987.0,1987,1987.0,earth orbit stations,karl buiter,1,sci-fi,"appii, c64",2,None
206,207,0,2005,2005.0,2005,2005.0,zoo tycoon ds,blue fang,1,modern,ds,1,first title in the series
174,175,0,2004,2004.0,2004,2004.0,holiday world,island,1,modern,win,1,None
167,168,0,2003,2003.0,2003,2003.0,x2: the threat,egosoft,1,sci-fi,"lin, osx, win",3,sequel to x: beyond the frontier
42,43,0,1994,1994.0,1994,1994.0,flamingo tours,sayonara,1,modern,"ami, dos",2,None
93,94,0,2000,2000.0,2000,2000.0,patrician ii: quest for power,ascaron,1,historical,win,1,sequel to the patrician
182,183,0,2004,2004.0,2004,2004.0,gamebiz,velocigames,1,modern,win,1,None
10,11,0,1983,1983.0,1983,1983.0,business games,acornsoft,1,modern,bbc,1,contains stokmark and telemark


In [26]:
durty_table.iloc[random_ids]

,Year,Game,Developer,Setting,Platform,Notes
11,1982,Corn Cropper,CCS,Modern,ZX,
145,2003,Car Tycoon,Vectorcom,Modern,WIN,
21,1987,Earth Orbit Stations,Karl Buiter,Sci-fi,"APPII, C64",
206,2005,Zoo Tycoon DS,Blue Fang,Modern,DS,First title in the series.
174,2004,Holiday World,Island,Modern,WIN,
167,2003,X2: The Threat,Egosoft,Sci-fi,"LIN, OSX, WIN",Sequel to X: Beyond the Frontier.
42,1994,Flamingo Tours,Sayonara,Modern,"AMI, DOS",
93,2000,Patrician II: Quest for Power,Ascaron,Historical,WIN,Sequel to The Patrician.
182,2004,GameBiz,Velocigames,Modern,WIN,
10,1983,Business Games,Acornsoft,Modern,BBC,Contains Stokmark and Telemark.
